In [2]:
import pandas as pd
import seaborn as srn
import statistics as sts

In [3]:
#Importar dados
dataset = pd.read_csv("data/Churn.csv", sep=";")
#Visualizar
dataset.head()

,X0,X1,X2,X3,X4,X4.1,X6,X7,X8,X9,X10,X11
0,1,619,RS,Feminino,42,2,0,1,1,1,10134888.0,1
1,2,608,SC,Feminino,41,1,8380786,1,0,1,11254258.0,0
2,3,502,RS,Feminino,42,8,1596608,3,1,0,11393157.0,1
3,4,699,RS,Feminino,39,1,0,2,0,0,9382663.0,0
4,5,850,SC,Feminino,43,2,12551082,1,1,1,790841.0,0


In [4]:
#Nomer as colunas
dataset.columns = ["Id", "Score", "Estado", "Genero", "Idade", "Parametro", "Saldo", "Produtos", "TemCartaoCredito", "Ativo", "Salario", "Saiu"] 

Tratar NAs na coluna Salario 

In [ ]:
#Remover NAs e substituir pela mediana
dataset['Salario'].describe()

count    9.920000e+02
mean     3.528762e+07
std      5.305800e+08
min      9.677000e+03
25%      3.029011e+06
50%      8.703250e+06
75%      1.405213e+07
max      1.193469e+10
Name: Salario, dtype: float64

In [ ]:
#Calcular a mediana do Salario
mediana = sts.median(dataset['Salario'])
mediana

70518.0

In [15]:
#Substituir NAs pela mediana
dataset['Salario'] = dataset['Salario'].fillna(mediana)

In [ ]:
#Verificar se existem NAs
dataset['Salario'].isnull().sum()

np.int64(0)

Padronuizar Genero

In [ ]:
#Agrupar dados
agrupado = dataset.groupby(['Genero']).size()
agrupado

Genero
F              2
Fem            1
Feminino     461
M              6
Masculino    521
dtype: int64

In [ ]:
#Verificar se existem NAs
dataset['Genero'].isnull().sum()

np.int64(8)

In [20]:
#Preencher NAs com Masculino (Moda)
dataset['Genero'] = dataset['Genero'].fillna('Masculino')

In [21]:
#Verificar se existem NAs
dataset['Genero'].isnull().sum()

np.int64(0)

In [22]:
#Padronização dos dominios
dataset.loc[dataset['Genero'] == 'M', 'Genero'] = "Masculino"
dataset.loc[dataset['Genero'].isin(['Fem', 'F']), 'Genero'] = "Feminino"
#Visualizar o resultado
agrupado = dataset.groupby(['Genero']).size()
agrupado

Genero
Feminino     464
Masculino    535
dtype: int64

Idades fora do dominio

In [24]:
#Verificar idades
dataset['Idade'].describe()

count    999.000000
mean      38.902903
std       11.401912
min      -20.000000
25%       32.000000
50%       37.000000
75%       44.000000
max      140.000000
Name: Idade, dtype: float64

In [ ]:
#Visualizar idades fora do dominio
dataset.loc[(dataset['Idade'] < 0) | (dataset['Idade'] > 120)]

,Id,Score,Estado,Genero,Idade,Parametro,Saldo,Produtos,TemCartaoCredito,Ativo,Salario,Saiu
867,869,636,RS,Feminino,-10,1,17083346,1,1,0,11051028.0,1
984,986,773,RS,Masculino,-20,1,12453278,2,0,1,1172357.0,0
990,992,655,RS,Masculino,140,5,93147,2,1,0,6621413.0,0


In [26]:
#Calcular a mediana
mediana = sts.median(dataset['Idade'])
mediana

37

In [27]:
#Substituir pela mediana
dataset.loc[(dataset['Idade'] < 0) | (dataset['Idade'] > 120), 'Idade'] = mediana

In [28]:
#Verificar idades fora do dominio
dataset.loc[(dataset['Idade'] < 0) | (dataset['Idade'] > 120)]

,Id,Score,Estado,Genero,Idade,Parametro,Saldo,Produtos,TemCartaoCredito,Ativo,Salario,Saiu


Dados duplicados

In [29]:
#Buscar pelo ID
dataset[dataset.duplicated(['Id'], keep=False)]

,Id,Score,Estado,Genero,Idade,Parametro,Saldo,Produtos,TemCartaoCredito,Ativo,Salario,Saiu
80,81,665,RS,Feminino,34,1,9664554,2,0,0,17141366.0,0
81,81,665,RS,Feminino,34,1,9664554,2,0,0,17141366.0,0


In [31]:
#Excluir pelo Id
dataset.drop_duplicates(subset="Id", keep='first', inplace=True)
#Buscar duplicidade
dataset[dataset.duplicated(['Id'], keep=False)]

,Id,Score,Estado,Genero,Idade,Parametro,Saldo,Produtos,TemCartaoCredito,Ativo,Salario,Saiu


Estados fora do dominio

In [32]:
#Agrupar dados
agrupado = dataset.groupby(['Estado']).size()
agrupado

Estado
PR    257
RP      1
RS    477
SC    258
SP      4
TD      1
dtype: int64

In [ ]:
#Substituir pelo RS (Moda)
dataset.loc[dataset['Estado'].isin(['RP', 'SP', 'TD']), 'Estado'] = "RS"

In [36]:
#Verificar resultado
agrupado = dataset.groupby(['Estado']).size()
agrupado

Estado
PR    257
RS    483
SC    258
dtype: int64

Outliers em salario

In [37]:
#Calcular desvio padrão
desv = sts.stdev(dataset['Salario'])
desv

528988918.4679201

In [ ]:
#Definir se uma salario for maior que 2 desvio padrão são considerados outliers
#Visualizar outliers
dataset.loc[dataset['Salario'] >= 2 * desv]

,Id,Score,Estado,Genero,Idade,Parametro,Saldo,Produtos,TemCartaoCredito,Ativo,Salario,Saiu
7,8,376,PR,Feminino,29,4,11504674,4,1,0,1.193469e+10,1
116,118,668,PR,Feminino,37,6,1678644,1,1,0,1.156383e+10,0
170,172,484,RS,Feminino,29,4,13011439,1,1,0,1.640179e+09,0
230,232,673,RS,Masculino,72,1,0,2,0,1,1.119812e+09,0


In [40]:
#Calcular mediana
mediana = sts.median(dataset['Salario'])
mediana

8637195.5

In [41]:
#Atualizar outliers para a mediana
dataset.loc[dataset['Salario'] >= 2 * desv, 'Salario'] = mediana
#Visualizar outliers
dataset.loc[dataset['Salario'] >= 2 * desv]

,Id,Score,Estado,Genero,Idade,Parametro,Saldo,Produtos,TemCartaoCredito,Ativo,Salario,Saiu
